# LLM API Comparison: Mistral vs Hugging Face
This notebook demonstrates how to query **Mistral** and **Hugging Face** APIs,
then compares their responses side by side.

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
# Store your API keys here — do NOT hardcode them in other cells.
# Better practice: use environment variables or a .env file.
import os
import requests

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "YOUR_MISTRAL_API_KEY_HERE")
HF_API_KEY      = os.getenv("HF_API_KEY",      "YOUR_HUGGINGFACE_API_KEY_HERE")

MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"
HF_URL      = "https://router.huggingface.co/hf-inference/models/google/flan-t5-small"

REQUEST_TIMEOUT = 30  # seconds

print("Configuration loaded.")

In [ ]:
# ─── Mistral API ──────────────────────────────────────────────────────────────
def query_mistral(prompt: str, model: str = "mistral-small") -> str:
    """
    Send a prompt to the Mistral chat completion API.
    Returns the assistant's reply as a string, or an error message.
    """
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}]
    }
    try:
        response = requests.post(MISTRAL_URL, json=payload, headers=headers, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()  # raises HTTPError for 4xx/5xx
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except requests.exceptions.HTTPError as e:
        return f"[Mistral HTTP Error] {e} — {response.text}"
    except requests.exceptions.Timeout:
        return "[Mistral Error] Request timed out."
    except (KeyError, IndexError):
        return f"[Mistral Error] Unexpected response structure: {response.json()}"
    except Exception as e:
        return f"[Mistral Error] {e}"


# Quick test
result = query_mistral("Explain deep learning in simple terms.")
print("Mistral response:\n")
print(result)

In [ ]:
# ─── Hugging Face Inference API ───────────────────────────────────────────────
def query_huggingface(prompt: str) -> str:
    """
    Send a prompt to the Hugging Face Inference API (flan-t5-small).
    Returns the generated text, or an error message.

    Note: flan-t5-small is a small seq2seq model — best for short factual
    Q&A. For richer answers, consider a larger model like flan-t5-xl.
    """
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}
    payload = {"inputs": prompt}
    try:
        response = requests.post(HF_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()
        # HF inference returns a list of dicts: [{"generated_text": "..."}]
        if isinstance(data, list) and data:
            return data[0].get("generated_text", "[No generated_text field]")
        elif isinstance(data, dict) and "error" in data:
            return f"[HuggingFace API Error] {data['error']}"
        return f"[HuggingFace Error] Unexpected format: {data}"
    except requests.exceptions.HTTPError as e:
        return f"[HuggingFace HTTP Error] {e} — {response.text}"
    except requests.exceptions.Timeout:
        return "[HuggingFace Error] Request timed out."
    except Exception as e:
        return f"[HuggingFace Error] {e}"


# Quick test
result = query_huggingface("What is the capital city of Indonesia?")
print("HuggingFace response:\n")
print(result)

In [ ]:
# ─── Comparison Function ──────────────────────────────────────────────────────
def compare_apis(prompt: str):
    """
    Query both APIs with the same prompt and print responses side by side.
    """
    print(f"Prompt: {prompt!r}")
    print("=" * 60)

    mistral_output    = query_mistral(prompt)
    huggingface_output = query_huggingface(prompt)

    print("\n[Mistral - mistral-small]")
    print("-" * 40)
    print(mistral_output)

    print("\n[HuggingFace - flan-t5-small]")
    print("-" * 40)
    print(huggingface_output)
    print()


compare_apis("Explain deep learning in simple terms.")